# Method 03 — Comparator-Composition Bootstrap and Placebo Ranking

Hold Australia's exact endpoint change fixed, resample eligible comparator countries, and locate Australia in the corresponding placebo ranking. Run Method 02 first to generate the frozen primary table.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SUBMISSION_ROOT = Path.cwd() / 'submission' if (Path.cwd() / 'submission' / 'OECD Data.csv').exists() else (Path.cwd().parent if Path.cwd().name == 'code' else Path.cwd())
sys.path.insert(0, str(SUBMISSION_ROOT.parent))
from submission.code.oecd_audit import INDICATOR_SPECS, load_clean

OUTCOME_SPECS = [('1_1', 2010, 2024, '2010–2024'), ('2_1', 2010, 2024, '2010–2024'),
                 ('7_1_DEP', 2010, 2024, '2008–10 to 2023–25 pooled windows'),
                 ('11_2', 2010, 2024, '2008–10 to 2023–25 pooled windows')]
SEED, REPLICATES = 20260720, 10_000
EXPECTED_COUNTS = [31, 43, 46, 46]
EXPECTED_GAPS = [-956.0, -0.864, -4.290870, -2.662420]
TABLE_DIR = SUBMISSION_ROOT / 'report' / 'tables'
FIGURE_DIR = SUBMISSION_ROOT / 'report' / 'figures'

def endpoint_changes(data, code, start, end):
    subset = data.loc[data.indicator_code.eq(code) & data.year.isin([start, end])].copy()
    subset = subset.sort_values('year').drop_duplicates(['country_code', 'independent_period'], keep='last')
    wide = (subset.pivot(index='country_code', columns='year', values='value')
            .reindex(columns=[start, end]).dropna().reset_index()
            .rename(columns={start: 'start_value', end: 'end_value'}))
    wide['native_change'] = wide.end_value - wide.start_value
    sign = 1 if INDICATOR_SPECS[code].direction == 'higher' else -1
    wide['oriented_change'] = wide.native_change * sign
    return wide

def bootstrap_gaps(australia_change, comparator_changes):
    rng = np.random.default_rng(SEED)
    return np.array([australia_change - np.median(rng.choice(comparator_changes,
                     size=len(comparator_changes), replace=True)) for _ in range(REPLICATES)])

def placebo_summary(changes):
    gaps = pd.DataFrame([(row.country_code, row.oriented_change - changes.loc[
        changes.country_code.ne(row.country_code), 'oriented_change'].median())
        for row in changes.itertuples(index=False)], columns=['country_code', 'gap'])
    australia_gap = gaps.loc[gaps.country_code.eq('AUS'), 'gap'].iloc[0]
    rank = gaps['gap'].rank(ascending=False, method='average').loc[gaps.country_code.eq('AUS')].iloc[0]
    others = gaps.loc[gaps.country_code.ne('AUS'), 'gap']
    return australia_gap, 100 * (len(gaps) - rank) / (len(gaps) - 1), int(others.gt(australia_gap).sum()), int(others.lt(australia_gap).sum()), int(others.eq(australia_gap).sum())

In [ ]:
primary_path = TABLE_DIR / 'material_social_primary_results.csv'
if not primary_path.exists():
    raise FileNotFoundError('Run submission/code/method02_primary_same_endpoint.ipynb first.')
data, primary = load_clean(), pd.read_csv(primary_path)
bootstrap_rows, placebo_rows = [], []
for spec, expected_count, expected_gap in zip(OUTCOME_SPECS, EXPECTED_COUNTS, EXPECTED_GAPS):
    code, start, end, period = spec
    changes = endpoint_changes(data, code, start, end)
    australia = changes.loc[changes.country_code.eq('AUS')].iloc[0]
    comparators = changes.loc[changes.country_code.ne('AUS'), 'oriented_change'].to_numpy()
    row = primary.loc[primary.indicator_code.eq(code)].iloc[0]
    observed_median = float(np.median(comparators))
    observed_gap = float(australia.oriented_change - observed_median)
    assert len(comparators) == expected_count and np.isclose(observed_gap, expected_gap, atol=1e-5)
    assert np.isclose(observed_gap, row.australia_minus_comparator_median_oriented, atol=1e-10)
    sampled = bootstrap_gaps(float(australia.oriented_change), comparators)
    lower, upper = np.quantile(sampled, [0.025, 0.975])
    crosses = bool(lower <= 0 <= upper)
    gap, percentile, more, less, tied = placebo_summary(changes)
    assert np.isclose(percentile, row.australia_favourable_percentile, atol=1e-10)
    bootstrap_rows.append({'indicator_code': code, 'indicator': row.indicator, 'unit': row.unit,
        'comparison_period': period, 'australia_oriented_change': australia.oriented_change,
        'observed_comparator_median_oriented_change': observed_median, 'observed_oriented_gap': observed_gap,
        'bootstrap_gap_median': float(np.median(sampled)), 'bootstrap_ci_lower': float(lower),
        'bootstrap_ci_upper': float(upper), 'interval_crosses_zero': crosses,
        'sensitivity_label': 'comparator-sensitive' if crosses else 'comparatively stable',
        'eligible_comparator_country_count': len(comparators), 'bootstrap_replicates': REPLICATES,
        'random_seed': SEED, 'uncertainty_scope': 'Comparator-country composition sensitivity; not OECD survey-sampling uncertainty.'})
    placebo_rows.append({'indicator_code': code, 'indicator': row.indicator, 'unit': row.unit,
        'comparison_period': period, 'australia_placebo_oriented_gap': gap,
        'australia_favourable_percentile': percentile, 'focal_country_count': len(changes),
        'eligible_comparator_country_count': len(comparators), 'more_favourable_country_count': more,
        'less_favourable_country_count': less, 'tied_country_count': tied,
        'percentile_definition': '0 = least favourable; 100 = most favourable; average ranks for ties',
        'evidence_relationship': 'Placebo rank and the Method 2 endpoint percentile are alternative presentations of the same comparison, not independent evidence.'})
bootstrap_results, placebo_results = pd.DataFrame(bootstrap_rows), pd.DataFrame(placebo_rows)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
bootstrap_path, placebo_path = TABLE_DIR / 'material_social_bootstrap_results.csv', TABLE_DIR / 'material_social_placebo_results.csv'
bootstrap_results.to_csv(bootstrap_path, index=False); placebo_results.to_csv(placebo_path, index=False)
pd.testing.assert_frame_equal(bootstrap_results, pd.read_csv(bootstrap_path), check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)
pd.testing.assert_frame_equal(placebo_results, pd.read_csv(placebo_path), check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)
display(bootstrap_results.round(3)); display(placebo_results.round(3))

In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plot = bootstrap_results.merge(placebo_results[['indicator_code', 'australia_favourable_percentile']], on='indicator_code')
names = {'1_1': 'Household income per person', '2_1': 'Employment rate', '7_1_DEP': 'Lack of social support', '11_2': 'Negative affect'}
units = {'1_1': 'USD per person, PPP', '2_1': 'Percentage points', '7_1_DEP': 'Percentage points', '11_2': 'Percentage points'}
fig, axes = plt.subplots(4, 1, figsize=(10, 9), constrained_layout=True)
for ax, row in zip(axes, plot.itertuples(index=False)):
    ax.axvline(0, color='#555555', linewidth=1)
    ax.errorbar(row.observed_oriented_gap, 0, xerr=[[row.observed_oriented_gap-row.bootstrap_ci_lower], [row.bootstrap_ci_upper-row.observed_oriented_gap]], fmt='o', color='#D55E00', ecolor='#0072B2', elinewidth=3, capsize=4)
    ax.set_yticks([]); ax.set_title(names[row.indicator_code], loc='left', fontweight='bold')
    ax.set_xlabel(f'Oriented Australia-minus-median gap ({units[row.indicator_code]}; positive favours Australia)')
    ax.text(.99, .82, f'Placebo percentile: {row.australia_favourable_percentile:.1f} | {row.eligible_comparator_country_count} comparators | {row.sensitivity_label}', transform=ax.transAxes, ha='right', va='top', fontsize=9)
fig.suptitle('Australia’s common-endpoint gaps: comparator-country sensitivity', fontweight='bold')
figure_path = FIGURE_DIR / 'material_social_comparative_gaps.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
assert figure_path.exists() and figure_path.stat().st_size > 0
plt.show(); print(f'Wrote {figure_path.relative_to(SUBMISSION_ROOT)} at 300 dpi')